
# Test Notebook — Heuristic Lower Agent

This notebook verifies the lower-level A3 heuristic controller.

Core rule to verify:

- **Positive offset** → handover harder → **stay / retain**
- **Negative offset** → handover easier → **go / offload**

The notebook tests:

1. Bias-to-offset sign logic.
2. Quantized A3 offsets.
3. Neutral behavior.
4. Neighbor load safety.
5. Multi-step load stabilization with a simplified load-transfer model.

Run all cells after placing the notebook in the same folder as:

- `heuristic_lower_agent.py`
- `local_a3_agent_wrapper.py`


In [ ]:

from pathlib import Path
import sys
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from heuristic_lower_agent import HeuristicLowerAgent
from local_a3_agent_wrapper import quantize_a3_offset

print('Loaded HeuristicLowerAgent from:', inspect.getfile(HeuristicLowerAgent))
print('Constructor signature:')
print(inspect.signature(HeuristicLowerAgent))



## 1. System setup

We use 3 gNBs and 3 slices. The neighbor graph is fully connected.


In [ ]:

GNB_IDS = [0, 1, 2]
GNB_NAMES = [f'gNB{i}' for i in GNB_IDS]
SLICE_TYPES = ['eMBB', 'URLLC', 'mMTC']
SLICE_INDEX = {s: k for k, s in enumerate(SLICE_TYPES)}

NEIGHBORS = {
    0: [1, 2],
    1: [0, 2],
    2: [0, 1],
}

VALID_OFFSETS = np.array([-6, -4, -2, 0, 2, 4, 6], dtype=float)

LOAD_TARGET = 0.60
KMAX = {s: 20.0 for s in SLICE_TYPES}


def load_matrix_to_dict(load_matrix):
    return {
        (gnb_id, slice_type): float(load_matrix[g, si])
        for g, gnb_id in enumerate(GNB_IDS)
        for si, slice_type in enumerate(SLICE_TYPES)
    }


def ue_counts_from_load(load_matrix, kmax=KMAX):
    # Simple proxy: convert normalized load to approximate UE count.
    return {
        (gnb_id, slice_type): int(round(float(load_matrix[g, si]) * kmax[slice_type]))
        for g, gnb_id in enumerate(GNB_IDS)
        for si, slice_type in enumerate(SLICE_TYPES)
    }


def make_agent(gnb_id, **kwargs):
    return HeuristicLowerAgent(
        gnb_id=gnb_id,
        neighbor_ids=NEIGHBORS[gnb_id],
        slice_types=SLICE_TYPES,
        **kwargs,
    )


def extract_offset(outputs, source, target, slice_type):
    return float(outputs[(source, target, slice_type)]['applied_offset_db'])


def extract_proto(outputs, source, target, slice_type):
    return float(outputs[(source, target, slice_type)]['proto_offset_db'])

print('Setup OK')



## 2. Utility plots


In [ ]:

def plot_heatmap(matrix, title, rows, cols, vmin=None, vmax=None, cmap=None):
    fig, ax = plt.subplots(figsize=(4.8, 3.2))
    im = ax.imshow(matrix, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f'{matrix[i, j]:.2f}', ha='center', va='center', color='white', fontweight='bold')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def offsets_to_target_matrix(offsets_by_agent, target_gnb):
    mat = np.full((len(GNB_IDS), len(SLICE_TYPES)), np.nan, dtype=float)
    for src in GNB_IDS:
        if src == target_gnb:
            continue
        outputs = offsets_by_agent[src]
        for si, s in enumerate(SLICE_TYPES):
            mat[src, si] = extract_offset(outputs, src, target_gnb, s)
    return mat



## 3. Test 1 — Basic sign logic

Expected behavior:

- Negative bias → negative offset.
- Positive bias → positive offset.
- Neutral bias → offset near 0.

For this test, the neighbor is safe: low load, low UE count, no handover failure, no ping-pong.


In [ ]:

def test_basic_sign_logic():
    load = np.array([
        [0.90, 0.60, 0.60],
        [0.20, 0.60, 0.60],
        [0.20, 0.60, 0.60],
    ], dtype=float)
    slice_loads = load_matrix_to_dict(load)
    ue_counts = ue_counts_from_load(load)

    cases = [
        {'case': 'offload', 'bias': {'eMBB': -1.0, 'URLLC': 0.0, 'mMTC': 0.0}, 'expected': 'negative'},
        {'case': 'retain',  'bias': {'eMBB': +1.0, 'URLLC': 0.0, 'mMTC': 0.0}, 'expected': 'positive'},
        {'case': 'neutral', 'bias': {'eMBB':  0.0, 'URLLC': 0.0, 'mMTC': 0.0}, 'expected': 'zero'},
    ]

    rows = []
    for c in cases:
        # For neutral, force neutral_offset_db=0 if the constructor supports it.
        agent = make_agent(0, neutral_offset_db=0.0)
        outputs = agent.compute_offsets(
            bias_row=c['bias'],
            ue_counts=ue_counts,
            kmax=KMAX,
            slice_loads=slice_loads,
        )
        for target in NEIGHBORS[0]:
            off = extract_offset(outputs, 0, target, 'eMBB')
            proto = extract_proto(outputs, 0, target, 'eMBB')
            if c['expected'] == 'negative':
                passed = off < 0
            elif c['expected'] == 'positive':
                passed = off > 0
            else:
                passed = abs(off) <= 0.1
            rows.append({
                'case': c['case'],
                'source': 'gNB0',
                'target': f'gNB{target}',
                'bias': c['bias']['eMBB'],
                'proto_offset': proto,
                'applied_offset': off,
                'expected': c['expected'],
                'PASS': passed,
            })
    return pd.DataFrame(rows)

sign_df = test_basic_sign_logic()
sign_df


In [ ]:

print('Basic sign logic PASS:', bool(sign_df['PASS'].all()))
assert sign_df['PASS'].all(), 'Basic sign logic failed. Check bias-to-offset sign mapping.'



## 4. Test 2 — Neighbor-load safety

Expected behavior:

If the serving gNB wants to offload but one neighbor is already overloaded, the heuristic should **avoid a negative offset** toward that overloaded neighbor.

Scenario:

- gNB0 eMBB is overloaded: `0.90`
- gNB1 eMBB is safe: `0.20`
- gNB2 eMBB is already overloaded: `0.90`

Expected:

- offset `gNB0 -> gNB1, eMBB` should be negative.
- offset `gNB0 -> gNB2, eMBB` should be zero or positive.


In [ ]:

def test_neighbor_load_safety():
    load = np.array([
        [0.90, 0.60, 0.60],
        [0.20, 0.60, 0.60],
        [0.90, 0.60, 0.60],
    ], dtype=float)
    slice_loads = load_matrix_to_dict(load)
    ue_counts = ue_counts_from_load(load)
    bias = {'eMBB': -1.0, 'URLLC': 0.0, 'mMTC': 0.0}

    # Try to enable load safety. If the heuristic ignores slice_loads or alpha_load,
    # this test will expose it.
    agent = make_agent(
        0,
        alpha_load=8.0,
        neutral_offset_db=0.0,
    )
    outputs = agent.compute_offsets(
        bias_row=bias,
        ue_counts=ue_counts,
        kmax=KMAX,
        slice_loads=slice_loads,
    )

    off_to_safe = extract_offset(outputs, 0, 1, 'eMBB')
    off_to_loaded = extract_offset(outputs, 0, 2, 'eMBB')

    rows = [
        {
            'source': 'gNB0',
            'target': 'gNB1',
            'target_load': load[1, 0],
            'applied_offset': off_to_safe,
            'expected': 'negative, because neighbor is safe',
            'PASS': off_to_safe < 0,
        },
        {
            'source': 'gNB0',
            'target': 'gNB2',
            'target_load': load[2, 0],
            'applied_offset': off_to_loaded,
            'expected': 'zero or positive, because neighbor is overloaded',
            'PASS': off_to_loaded >= 0,
        },
    ]
    return pd.DataFrame(rows), outputs

neighbor_safety_df, neighbor_safety_outputs = test_neighbor_load_safety()
neighbor_safety_df


In [ ]:

print('Neighbor-load safety PASS:', bool(neighbor_safety_df['PASS'].all()))
if not neighbor_safety_df['PASS'].all():
    print('WARNING: The heuristic is not blocking offload toward an overloaded neighbor.')
    print('This usually means slice_loads or alpha_load are not used in the offset formula.')



## 5. Test 3 — Mobility safety

Expected behavior:

If a neighbor direction has high handover failure ratio or high ping-pong ratio, the offset should become less negative or positive.


In [ ]:

def test_mobility_safety():
    load = np.array([
        [0.90, 0.60, 0.60],
        [0.20, 0.60, 0.60],
        [0.20, 0.60, 0.60],
    ], dtype=float)
    slice_loads = load_matrix_to_dict(load)
    ue_counts = ue_counts_from_load(load)
    bias = {'eMBB': -1.0, 'URLLC': 0.0, 'mMTC': 0.0}

    agent = make_agent(0, neutral_offset_db=0.0)

    safe_outputs = agent.compute_offsets(
        bias_row=bias,
        ue_counts=ue_counts,
        kmax=KMAX,
        slice_loads=slice_loads,
        handover_failure_ratios={(0, 1, 'eMBB'): 0.0, (0, 2, 'eMBB'): 0.0},
        ping_pong_ratios={(0, 1, 'eMBB'): 0.0, (0, 2, 'eMBB'): 0.0},
    )
    safe_off = extract_offset(safe_outputs, 0, 1, 'eMBB')

    agent.reset()
    risky_outputs = agent.compute_offsets(
        bias_row=bias,
        ue_counts=ue_counts,
        kmax=KMAX,
        slice_loads=slice_loads,
        handover_failure_ratios={(0, 1, 'eMBB'): 0.90},
        ping_pong_ratios={(0, 1, 'eMBB'): 0.80},
    )
    risky_off = extract_offset(risky_outputs, 0, 1, 'eMBB')

    df = pd.DataFrame([
        {'case': 'safe neighbor', 'applied_offset': safe_off},
        {'case': 'risky neighbor', 'applied_offset': risky_off},
    ])
    passed = risky_off >= safe_off
    return df, passed

mobility_df, mobility_pass = test_mobility_safety()
display(mobility_df)
print('Mobility safety PASS:', mobility_pass)
assert mobility_pass, 'Mobility safety failed: risky neighbor should not receive more aggressive negative offset.'



## 6. Full 3-gNB heuristic snapshot

This reproduces the kind of plot you showed:

- load before
- bias
- offsets to each target gNB
- load after a simplified stabilization simulation


In [ ]:

def compute_rule_based_global_bias(load, target=LOAD_TARGET, margin=0.10):
    bias = np.zeros_like(load, dtype=float)
    bias[load > target + margin] = -1.0
    bias[load < target - margin] = +1.0
    return bias


def compute_all_offsets(load, bias, agent_kwargs=None):
    agent_kwargs = agent_kwargs or {}
    slice_loads = load_matrix_to_dict(load)
    ue_counts = ue_counts_from_load(load)
    offsets_by_agent = {}
    details = []

    for src in GNB_IDS:
        agent = make_agent(src, **agent_kwargs)
        bias_row = {s: float(bias[src, si]) for si, s in enumerate(SLICE_TYPES)}
        outputs = agent.compute_offsets(
            bias_row=bias_row,
            ue_counts=ue_counts,
            kmax=KMAX,
            slice_loads=slice_loads,
        )
        offsets_by_agent[src] = outputs
        for key, info in outputs.items():
            i, j, s = key
            details.append({
                'source': f'gNB{i}',
                'target': f'gNB{j}',
                'slice': s,
                **info,
            })
    return offsets_by_agent, pd.DataFrame(details)


initial_load = np.array([
    [0.80, 0.20, 0.20],
    [0.20, 0.20, 0.20],
    [0.20, 0.20, 0.20],
], dtype=float)

bias = compute_rule_based_global_bias(initial_load)

offsets_by_agent, offset_details_df = compute_all_offsets(
    initial_load,
    bias,
    agent_kwargs=dict(alpha_load=8.0, neutral_offset_db=0.0),
)

display(offset_details_df)

plot_heatmap(initial_load, 'load before', GNB_NAMES, SLICE_TYPES, vmin=0, vmax=1)
plot_heatmap(bias, 'global bias', GNB_NAMES, SLICE_TYPES, vmin=-1, vmax=1)
for target in GNB_IDS:
    mat = offsets_to_target_matrix(offsets_by_agent, target)
    plot_heatmap(mat, f'offsets to gNB{target}', GNB_NAMES, SLICE_TYPES, vmin=-6, vmax=6)



## 7. Simplified load-transfer simulation

This is not the full radio simulator. It is only a sanity model to check whether the heuristic can reduce load imbalance without emptying the source or saturating neighbors.

Transfer rule:

- Only overloaded source gNBs can send traffic.
- Only neighbors below target can receive traffic.
- Negative offset controls transfer strength.
- Transfer is capped by source excess and target capacity.


In [ ]:

def simulate_one_step(load, bias, agent_kwargs=None, max_transfer=0.04, target=LOAD_TARGET):
    load = load.copy().astype(float)
    offsets_by_agent, _ = compute_all_offsets(load, bias, agent_kwargs=agent_kwargs)
    delta = np.zeros_like(load)

    for src in GNB_IDS:
        for si, s in enumerate(SLICE_TYPES):
            source_excess = max(load[src, si] - target, 0.0)
            if source_excess <= 1e-9:
                continue

            candidates = []
            for dst in NEIGHBORS[src]:
                offset = extract_offset(offsets_by_agent[src], src, dst, s)
                dst_capacity = max(target - load[dst, si], 0.0)
                if offset < 0 and dst_capacity > 0:
                    strength = min(abs(offset) / 6.0, 1.0)
                    candidates.append((dst, strength, dst_capacity, offset))

            if not candidates:
                continue

            total_weight = sum(strength * cap for _, strength, cap, _ in candidates)
            if total_weight <= 1e-9:
                continue

            max_total_move = min(source_excess, max_transfer)
            for dst, strength, cap, offset in candidates:
                weight = strength * cap / total_weight
                amount = min(max_total_move * weight, cap)
                delta[src, si] -= amount
                delta[dst, si] += amount

    new_load = np.clip(load + delta, 0.0, 1.0)
    return new_load, offsets_by_agent, delta


def run_stabilization(load0, steps=30, agent_kwargs=None, target=LOAD_TARGET):
    load = load0.copy().astype(float)
    history = [load.copy()]
    bias_history = []
    var_history = []
    overload_history = []

    for t in range(steps):
        bias = compute_rule_based_global_bias(load, target=target, margin=0.05)
        bias_history.append(bias.copy())
        var_history.append(float(np.mean(np.var(load, axis=0))))
        overload_history.append(int(np.sum(load > target + 0.10)))
        load, offsets, delta = simulate_one_step(load, bias, agent_kwargs=agent_kwargs, target=target)
        history.append(load.copy())

    var_history.append(float(np.mean(np.var(load, axis=0))))
    overload_history.append(int(np.sum(load > target + 0.10)))
    return np.asarray(history), bias_history, np.asarray(var_history), np.asarray(overload_history)

history, bias_history, var_history, overload_history = run_stabilization(
    initial_load,
    steps=30,
    agent_kwargs=dict(alpha_load=8.0, neutral_offset_db=0.0),
)

final_load = history[-1]
plot_heatmap(final_load, 'load after stabilization', GNB_NAMES, SLICE_TYPES, vmin=0, vmax=1)
print('Initial eMBB loads:', history[0, :, 0])
print('Final eMBB loads:  ', history[-1, :, 0])
print('Variance before:', var_history[0])
print('Variance after: ', var_history[-1])


In [ ]:

plt.figure(figsize=(7, 4))
for gi, name in enumerate(GNB_NAMES):
    plt.plot(history[:, gi, SLICE_INDEX['eMBB']], marker='o', label=f'{name} eMBB')
plt.axhline(LOAD_TARGET, linestyle='--', label='target')
plt.title('eMBB load evolution')
plt.xlabel('step')
plt.ylabel('load')
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(var_history, marker='o')
plt.title('Mean slice-load variance over gNBs')
plt.xlabel('step')
plt.ylabel('variance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(overload_history, marker='o')
plt.title('Overload count')
plt.xlabel('step')
plt.ylabel('count')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



## 8. Final acceptance checks

The heuristic is considered acceptable for this simplified test if:

1. Mean load variance decreases.
2. No gNB-slice is saturated at `1.0`.
3. The overloaded source does not become empty.
4. Neighbor-load safety test passes.


In [ ]:

acceptance = []
acceptance.append({
    'check': 'variance decreased',
    'value': f'{var_history[0]:.4f} -> {var_history[-1]:.4f}',
    'PASS': bool(var_history[-1] < var_history[0]),
})
acceptance.append({
    'check': 'no saturation',
    'value': f'max final load = {final_load.max():.3f}',
    'PASS': bool(final_load.max() < 0.99),
})
acceptance.append({
    'check': 'source not emptied',
    'value': f'final gNB0 eMBB = {final_load[0, 0]:.3f}',
    'PASS': bool(final_load[0, 0] > 0.05),
})
acceptance.append({
    'check': 'neighbor-load safety',
    'value': str(bool(neighbor_safety_df['PASS'].all())),
    'PASS': bool(neighbor_safety_df['PASS'].all()),
})

acceptance_df = pd.DataFrame(acceptance)
display(acceptance_df)

if acceptance_df['PASS'].all():
    print('FINAL RESULT: PASS — heuristic behavior is stable in this sanity notebook.')
else:
    print('FINAL RESULT: WARNING — at least one sanity check failed.')
    print('Most common fix: update HeuristicLowerAgent so slice_loads affects the safety term, and set neutral_offset_db=0.')



## Conclusion

Use this notebook as a quick sanity check before PPO/TD3 training.

The lower heuristic should produce:

- `bias < 0` → negative offset only toward safe neighbors.
- `bias > 0` → positive offset.
- `bias = 0` → near-zero offset.
- overloaded neighbor → zero or positive offset.

If the notebook fails the neighbor-load safety test, the heuristic probably receives `slice_loads` but does not use them in the offset formula.
